### Machine Learning

A complete **Linear Regression** workflow: load the data, split it into train/test sets, fit the model, and evaluate it.

In [1]:
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

## 1. Load the data

`load_diabetes` is bundled with scikit-learn, so it needs no download.
Each row is a patient; the 10 features are baseline measurements and `target`
is disease progression one year later.

In [2]:
dataset = load_diabetes(as_frame=True)
df = dataset.frame

print(f'Rows: {df.shape[0]} | Columns: {df.shape[1]}')
df.head()

Rows: 442 | Columns: 11


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [3]:
X = dataset.data
y = dataset.target

print(f'Feature matrix: {X.shape}')
print(f'Target vector : {y.shape}')
print('\nFeatures:')
for column in X.columns:
    print(f'  - {column}')

Feature matrix: (442, 10)
Target vector : (442,)

Features:
  - age
  - sex
  - bmi
  - bp
  - s1
  - s2
  - s3
  - s4
  - s5
  - s6


## 2. Split into train and test sets

The model is fitted on 80% of the rows and scored on the 20% it has never seen,
which is what makes the evaluation meaningful. `random_state` keeps the split
reproducible.

In [4]:
TEST_SIZE = 0.2
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print(f'Training rows: {X_train.shape[0]}')
print(f'Testing rows : {X_test.shape[0]}')

Training rows: 353
Testing rows : 89


## 3. Train the model

`fit` finds the coefficients that minimise the mean squared error on the
training set.

In [5]:
model = LinearRegression()
model.fit(X_train, y_train)

print(f'Intercept: {model.intercept_:.2f}')
print('\nCoefficients:')
for name, coef in zip(X.columns, model.coef_):
    print(f'  {name:<6} {coef:>10.2f}')

Intercept: 151.35

Coefficients:
  age         37.90
  sex       -241.96
  bmi        542.43
  bp         347.70
  s1        -931.49
  s2         518.06
  s3         163.42
  s4         275.32
  s5         736.20
  s6          48.67


## 4. Evaluate

Predictions are made on the held-out test set. MAE and RMSE are in the same
units as the target, while R² is the share of the target's variance the model
explains. Comparing train R² against test R² shows whether the model is
overfitting.

In [6]:
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = mse ** 0.5
r2 = r2_score(y_test, predictions)

print(f'MAE  : {mae:.2f}')
print(f'MSE  : {mse:.2f}')
print(f'RMSE : {rmse:.2f}')
print(f'R^2  : {r2:.4f}')
print()
print(f'Train R^2: {model.score(X_train, y_train):.4f}')
print(f'Test  R^2: {r2:.4f}')

MAE  : 42.79
MSE  : 2900.19
RMSE : 53.85
R^2  : 0.4526

Train R^2: 0.5279
Test  R^2: 0.4526


In [7]:
comparison = pd.DataFrame({'actual': y_test.to_numpy(), 'predicted': predictions})
comparison['residual'] = comparison['actual'] - comparison['predicted']
comparison.head(10)

,actual,predicted,residual
0,219.0,139.547558,79.452442
1,70.0,179.517208,-109.517208
2,202.0,134.038756,67.961244
3,230.0,291.417029,-61.417029
4,111.0,123.789659,-12.789659
5,84.0,92.172347,-8.172347
6,242.0,258.232389,-16.232389
7,272.0,181.337321,90.662679
8,94.0,90.224113,3.775887
9,96.0,108.633759,-12.633759
